# Multi-Session LeJEPA Experiment Runner

Train and evaluate the LeJEPA teacher model on multi-session neural spike data.

**Order of operations:**
1. Set `CONFIG_PATH` in the Config cell.
2. Run all cells top to bottom.
3. Results (CSV + plots) are saved under `results_out_path` from your config.
4. Training curves, test metrics, and latent-space UMAP also render inline below.

**Prerequisite:** Run the dataset pipeline first (`experiments/data/create_dataset.py`) so a Parquet exists at your config's `data_path`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/jacobposchl/jepsyn
!pip install snntorch temporaldata torch_brain umap-learn pyarrow pyyaml

# ensure the datasets directory exists and copy the parquet file from the drive to the datasets directory
# modify the path to the parquet file as needed
!mkdir -p /content/jepsyn/datasets
!cp "/content/drive/MyDrive/SNN-LeJEPA/Data/visp_visl_novel/visp_visl_windows.parquet" /content/jepsyn/datasets/

In [ ]:
import os
import sys
from pathlib import Path

# The repo is cloned to /content/jepsyn in Colab.
REPO_ROOT = Path("/content/jepsyn")
os.chdir(REPO_ROOT)
print(f"CWD: {Path.cwd()}")

# jepsyn/ package is importable directly from repo root.
# Add experiments/multi_session/ so multi_session.py can be imported by name.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "experiments" / "multi_session"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from multi_session import evaluate_model, identify_units, load_and_prepare_data, save_results, train_lejepa
from jepsyn.utils import verify_config

print("Imports OK")

## Sanity Checks (POYO Encoder)

Run this cell before training to verify the POYO refactor is wired correctly.
Tests: forward pass shapes, delimiter injection counts, gradient flow, unit dropout, eval determinism.
No GPU or real dataset required — uses small dummy data.

In [ ]:
# =============================================================================
# POYO Encoder Sanity Checks
# Verifies the POYO refactor before training. No GPU or real dataset required.
# =============================================================================
import torch
from jepsyn.models import NeuralEncoder
from jepsyn.utils import apply_unit_dropout

print("=" * 60)
print("POYO Encoder Sanity Checks")
print("=" * 60)

# --- Dummy session setup ---
# Session 0: 3 units (raw IDs 10-12, mapped to 1-3)
# Session 1: 4 units (raw IDs 20-23, mapped to 1-4)
session_unit_maps = {
    0: {10: 1, 11: 2, 12: 3},
    1: {20: 1, 21: 2, 22: 3, 23: 4},
}

enc = NeuralEncoder(
    session_unit_maps=session_unit_maps,
    d_model=32,
    n_latents=8,
    window_size_s=0.4,
    n_cross_attn_heads=4,
    n_self_attn_heads=4,
    n_self_attn_layers=2,
    dim_feedforward=64,
    dropout=0.0,
    rope_t_min=1e-3,
    rope_t_max=4.0,
    use_delimiter_tokens=True,
)

B, E = 3, 5
session_ids = torch.tensor([0, 0, 1])
# sample 0: 2 real spikes (units 1,2)  sample 1: 3 real spikes (units 1,3,2)  sample 2: 4 real spikes (units 1,2,3,4)
unit_ids  = torch.tensor([[1, 2, 0, 0, 0],
                           [1, 3, 2, 0, 0],
                           [1, 2, 3, 4, 0]], dtype=torch.long)
time_ids  = torch.tensor([[ 50, 200,   0,   0,   0],
                           [ 10, 150, 300,   0,   0],
                           [100, 180, 250, 350,   0]], dtype=torch.long)
attn_mask = torch.tensor([[True,  True,  False, False, False],
                           [True,  True,  True,  False, False],
                           [True,  True,  True,  True,  False]])

# ---- Test 1: Forward pass shapes ----
enc.eval()
with torch.no_grad():
    Z, h = enc(session_ids, unit_ids, time_ids, attn_mask)

assert Z.shape == (B, 8, 32), f"Z shape wrong: {Z.shape}"
assert h.shape == (B, 32),    f"h shape wrong: {h.shape}"
print(f"Test 1 PASS — Forward pass shapes correct: Z={tuple(Z.shape)}, h={tuple(h.shape)}")

# ---- Test 2: Delimiter injection counts ----
# Session 0: 3 units → 6 delimiter tokens (START+END for each)
# Session 1: 4 units → 8 delimiter tokens
# max_delim = 8, so total seq_len after injection = E + 8 = 13
enc.train()
x_unit = torch.zeros(B, E, 32)
x_comb, ts_s, mask_comb = enc.encoder._inject_delimiters(x_unit, time_ids, session_ids, attn_mask)
expected_seq_len = E + 2 * max(len(session_unit_maps[0]), len(session_unit_maps[1]))  # 5+8=13
assert x_comb.shape[1] == expected_seq_len, \
    f"Expected seq_len={expected_seq_len}, got {x_comb.shape[1]}"
n_delim_s0 = mask_comb[0, E:].sum().item()   # session 0: 3 units → 6 delimiters
n_delim_s2 = mask_comb[2, E:].sum().item()   # session 1: 4 units → 8 delimiters
assert n_delim_s0 == 6, f"Session 0: expected 6 delimiter tokens, got {n_delim_s0}"
assert n_delim_s2 == 8, f"Session 1: expected 8 delimiter tokens, got {n_delim_s2}"
print(f"Test 2 PASS — Delimiter counts: sess0={int(n_delim_s0)}/6, sess1={int(n_delim_s2)}/8  "
      f"(total seq_len={x_comb.shape[1]})")

# ---- Test 3: Gradient flow through unit_embeds, session_embed, delimiter_embed ----
enc.train()
for p in enc.parameters():
    if p.grad is not None:
        p.grad.zero_()
Z, h = enc(session_ids, unit_ids, time_ids, attn_mask)
h.sum().backward()
unit_grad  = enc.encoder.unit_embeds["0"].weight.grad
sess_grad  = enc.encoder.session_embed.weight.grad
delim_grad = enc.encoder.delimiter_embed.weight.grad
assert unit_grad  is not None and unit_grad.abs().sum()  > 0, "unit_embeds[0] has no gradient"
assert sess_grad  is not None and sess_grad.abs().sum()  > 0, "session_embed has no gradient"
assert delim_grad is not None and delim_grad.abs().sum() > 0, "delimiter_embed has no gradient"
print("Test 3 PASS — Gradients flow to unit_embeds, session_embed, delimiter_embed")

# ---- Test 4: apply_unit_dropout ----
unit_ids_t = torch.tensor([[1, 1, 2, 2, 3],
                             [1, 2, 3, 4, 4]], dtype=torch.long)
mask_t     = torch.ones(2, 5, dtype=torch.bool)
dropped    = apply_unit_dropout(unit_ids_t, mask_t, dropout_ratio=0.5, min_units=1)
assert dropped.shape == mask_t.shape, "apply_unit_dropout changed mask shape"
for i in range(2):
    remaining = unit_ids_t[i][dropped[i]].unique()
    assert len(remaining) >= 1, f"Row {i}: all units were dropped (min_units violated)"
print(f"Test 4 PASS — apply_unit_dropout: real tokens remaining {dropped.sum(dim=1).tolist()} "
      f"(from {mask_t.sum(dim=1).tolist()})")

# ---- Test 5: Eval-mode determinism ----
enc.eval()
with torch.no_grad():
    _, h1 = enc(session_ids, unit_ids, time_ids, attn_mask)
    _, h2 = enc(session_ids, unit_ids, time_ids, attn_mask)
assert torch.allclose(h1, h2), "Encoder output differs between two eval-mode runs!"
print("Test 5 PASS — Eval mode is deterministic")

print()
print("All sanity checks passed. Ready to train!")

In [ ]:
!export PYTHONPATH=$PYTHONPATH:. && python experiments/multi_session/multi_session.py experiments/multi_session/configs/ablation_vicreg.yaml

In [ ]:
!export PYTHONPATH=$PYTHONPATH:. && python experiments/multi_session/multi_session.py experiments/multi_session/configs/ablation_poyo.yaml

In [ ]:
!export PYTHONPATH=$PYTHONPATH:. && python experiments/multi_session/multi_session.py experiments/multi_session/configs/ablation_mae.yaml

In [ ]:
!zip -r results_export.zip results

In [ ]:
from google.colab import files
files.download('results_export.zip')